# Bible Corpus: Pairwise Domain Distance

Downloads selected language bibles from https://github.com/christos-c/bible-corpus and computes
pairwise domain distances using the metrics in `corpus_helpers.metrics`.

**Languages chosen** (mix of close and distant pairs):
- **Germanic**: `en` (English), `de` (German), `nl` (Dutch)
- **Slavic**: `cs` (Czech), `sk` (Slovak), `pl` (Polish)
- **Romance**: `fr` (French)

Expected clusters: {de, nl, en}, {cs, sk, pl}, with fr farther from both.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

from corpus_helpers.metrics import (
    ngram_overlap,
    ngram_divergence,
    normalized_compression_distance,
)

## 1. Download Bible XML files

In [ ]:
LANGUAGES = {
    'en': 'English',
    'de': 'German',
    'nl': 'Dutch',
    'cs': 'Czech',
    'sk': 'Slovak',
    'pl': 'Polish',
    'fr': 'French',
}

RAW_BASE = 'https://raw.githubusercontent.com/christos-c/bible-corpus/master/bibles'
CACHE_DIR = Path('_bible_cache')
CACHE_DIR.mkdir(exist_ok=True)

def download(lang_code: str, lang_name: str) -> Path:
    dest = CACHE_DIR / f'{lang_code}.xml'
    if not dest.exists():
        url = f'{RAW_BASE}/{lang_name}.xml'
        print(f'Downloading {lang_name}...', end=' ', flush=True)
        urllib.request.urlretrieve(url, dest)
        print('done')
    else:
        print(f'{lang_name}: cached')
    return dest

paths = {code: download(code, name) for code, name in LANGUAGES.items()}

## 2. Parse XML → list of verse bytes

In [ ]:
def parse_bible(path: Path) -> list[bytes]:
    """Extract all <seg> text nodes as UTF-8 byte strings."""
    tree = ET.parse(path)
    verses = [
        (seg.text or '').strip().encode('utf-8')
        for seg in tree.iter('seg')
        if seg.text and seg.text.strip()
    ]
    return verses

corpora: dict[str, list[bytes]] = {}
for code, path in paths.items():
    corpora[code] = parse_bible(path)
    total_bytes = sum(len(v) for v in corpora[code])
    print(f'{code}: {len(corpora[code]):,} verses, {total_bytes/1e6:.2f} MB')

## 3. Compute pairwise distance matrices

Three metrics:
- **NCD** – Normalized Compression Distance (zlib, symmetric). Lower = more similar.
- **JSD** – Jensen-Shannon Divergence on word unigrams. Lower = more similar.
- **Jaccard distance** – 1 − Jaccard overlap of word-unigram vocabularies. Lower = more similar.

In [ ]:
LANG_CODES = list(LANGUAGES.keys())
N = len(LANG_CODES)

def empty_matrix() -> np.ndarray:
    return np.zeros((N, N))

ncd_mat = empty_matrix()
jsd_mat = empty_matrix()
jaccard_dist_mat = empty_matrix()

pairs = list(combinations(range(N), 2))
print(f'Computing {len(pairs)} pairs × 3 metrics...')

for i, j in pairs:
    ca, cb = corpora[LANG_CODES[i]], corpora[LANG_CODES[j]]
    label = f"{LANG_CODES[i]}-{LANG_CODES[j]}"
    print(f'  {label}', end='  ', flush=True)

    ncd = normalized_compression_distance(ca, cb, symmetric=True)
    jsd = ngram_divergence(ca, cb, method='jsd', n=1, unit='word')
    jac = 1.0 - ngram_overlap(ca, cb, n=1, unit='word')
    print(f'NCD={ncd:.4f}  JSD={jsd:.4f}  Jaccard-dist={jac:.4f}')

    for mat, val in [(ncd_mat, ncd), (jsd_mat, jsd), (jaccard_dist_mat, jac)]:
        mat[i, j] = mat[j, i] = val

print('Done.')

## 4. Visualise: Heatmaps

In [ ]:
lang_labels = [f"{c} ({LANGUAGES[c]})" for c in LANG_CODES]

METRIC_INFO = [
    (ncd_mat,          'NCD (zlib, symmetric)',          'YlOrRd'),
    (jsd_mat,          'JSD word unigrams',              'YlOrRd'),
    (jaccard_dist_mat, 'Jaccard distance (word vocab)',  'YlOrRd'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Pairwise domain distance across languages (Bible corpus)', fontsize=14)

for ax, (mat, title, cmap) in zip(axes, METRIC_INFO):
    mask = np.eye(N, dtype=bool)  # hide diagonal (zero)
    sns.heatmap(
        mat,
        ax=ax,
        annot=True,
        fmt='.3f',
        cmap=cmap,
        xticklabels=LANG_CODES,
        yticklabels=LANG_CODES,
        mask=mask,
        linewidths=0.5,
        vmin=0,
    )
    ax.set_title(title)

plt.tight_layout()
plt.savefig('bible_domain_distance_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Visualise: Dendrograms

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Hierarchical clustering of languages by domain distance (Bible corpus)', fontsize=14)

for ax, (mat, title, _) in zip(axes, METRIC_INFO):
    condensed = squareform(mat)
    Z = linkage(condensed, method='average')
    dendrogram(
        Z,
        labels=LANG_CODES,
        ax=ax,
        leaf_rotation=45,
        color_threshold=0.7 * max(Z[:, 2]),
    )
    ax.set_title(title)
    ax.set_ylabel('distance')

plt.tight_layout()
plt.savefig('bible_domain_distance_dendrograms.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary table

In [ ]:
rows = []
for i, j in combinations(range(N), 2):
    rows.append({
        'pair': f"{LANG_CODES[i]}-{LANG_CODES[j]}",
        'NCD': ncd_mat[i, j],
        'JSD': jsd_mat[i, j],
        'Jaccard_dist': jaccard_dist_mat[i, j],
    })

df = pd.DataFrame(rows).set_index('pair').sort_values('NCD')
print('Pairs sorted by NCD (ascending = more similar):')
df.style.background_gradient(cmap='YlOrRd', axis=0)